# D2-05 Regionalised methods across impact categories

This notebook compares two BAFU tomato systems with the same functional unit: **1 kg fresh tomatoes**.

- **Swiss heated greenhouse**: early-harvest tomatoes produced in Switzerland
- **Spanish greenhouse**: greenhouse tomatoes produced in Spain

We use two regionalised methods to ask a simple question: **how do the results change when water use and land-related pressures occur in different places?**

## Learning goals

After this notebook, you should be able to:

- calculate two regionalised impact categories for comparable products
- compare results only when they belong to the same impact category
- identify the processes and locations that contribute most to a score
- interpret signed water-scarcity contributions and land-use biodiversity contributions
- decompose vertebrate IBIF results into climate change, land use, and roads
- reuse an `EdgeLCIA` object with `redo_lcia()` when the method stays fixed

## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023
- Schipper, A. M., et al. (2025). Intactness-based biodiversity impact factors for life cycle assessment. *Scientific Data*. https://doi.org/10.1038/s41597-025-05946-1

## 1) Select two comparable tomato datasets

We identify activities by their **name**, **reference product**, and **location**. The helper raises an error unless exactly one activity matches, which prevents us from silently selecting the wrong dataset.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bw2data as bd
from edges import EdgeLCIA, get_available_methods

pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

In [ ]:
bd.projects.set_current("aalborg-rlcia-2026")

if "bafu" not in bd.databases:
    raise ValueError("The 'bafu' database is missing. Complete the Day 1 import first.")

db = bd.Database("bafu")
print("Project:", bd.projects.current)
print("Database:", db.name)

In [ ]:
def find_activity(database, name, product, location):
    matches = [
        activity for activity in database
        if activity.get("name") == name
        and activity.get("reference product") == product
        and activity.get("location") == location
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one match for {name!r} in {location}; found {len(matches)}")
    return matches[0]


swiss_name = "Tomatoes conventional, hors-sol heated, at greenhouse, early harvest"
spanish_name = "Tomatoes conventional, hors-sol, at greenhouse"

tomatoes = {
    "Swiss heated greenhouse": find_activity(db, swiss_name, swiss_name, "CH"),
    "Spanish greenhouse": find_activity(db, spanish_name, spanish_name, "ES"),
}

tomato_table = pd.DataFrame([
    {
        "Tomato system": label,
        "Dataset": activity["name"],
        "Location": activity["location"],
        "Functional unit": f"1 {activity['unit']}",
    }
    for label, activity in tomatoes.items()
])
tomato_table

## 2) Define two regionalised methods

We compare two different environmental questions:

- **AWARE 2.0 water scarcity** responds to where water consumption occurs.
- **IBIF land use** responds to where land occupation affects terrestrial biodiversity.

The scores have different units and must **not** be compared across impact categories.

In [ ]:
aware_method = ("AWARE 2.0", "Country", "all", "yearly")
ibif_landuse_method = ("IBIF", "biodiversity", "LU", "overall")

methods = {
    "AWARE water scarcity": {
        "method": aware_method,
        "unit": "m³ deprived water-eq.",
        "color": "#2c7fb8",
    },
    "IBIF land use": {
        "method": ibif_landuse_method,
        "unit": "MSA-loss·km²·yr",
        "color": "#2e8b57",
    },
}

In [ ]:
available_methods = set(get_available_methods())
missing_methods = [
    info["method"] for info in methods.values()
    if info["method"] not in available_methods
]
if missing_methods:
    raise ValueError(f"These methods are unavailable: {missing_methods}")

method_table = pd.DataFrame([
    {"Impact category": label, "Unit": info["unit"], "Method tuple": info["method"]}
    for label, info in methods.items()
])
method_table

## 3) Calculate and compare the scores

For each combination of tomato system and method, we follow the same four-step workflow. We create a fresh `EdgeLCIA` object each time so the calculation remains easy to follow. Section 8 introduces the faster `redo_lcia()` pattern.

In [ ]:
def run_edges(activity, method, amount=1):
    analysis = EdgeLCIA({activity: amount}, method)
    analysis.lci()
    analysis.apply_strategies()
    analysis.evaluate_cfs()
    analysis.lcia()
    return analysis

In [ ]:
analyses = {}
cf_tables = {}
score_rows = []

for tomato_label, activity in tomatoes.items():
    for impact_label, method_info in methods.items():
        print(f"Calculating {impact_label} for {tomato_label} ...")
        analysis = run_edges(activity, method_info["method"])

        key = (tomato_label, impact_label)
        analyses[key] = analysis
        cf_tables[key] = analysis.generate_cf_table(split_aggregate_consumers=True)
        score_rows.append({
            "Tomato system": tomato_label,
            "Impact category": impact_label,
            "Score": float(analysis.score),
            "Unit": method_info["unit"],
        })

scores = pd.DataFrame(score_rows)
display(scores.style.format({"Score": "{:.3g}"}))

In [ ]:
comparison_rows = []

for impact_label, method_info in methods.items():
    category_scores = scores[scores["Impact category"] == impact_label]
    category_scores = category_scores.set_index("Tomato system")["Score"]

    larger_system = category_scores.idxmax()
    ratio = category_scores.max() / category_scores.min()
    comparison_rows.append({
        "Impact category": impact_label,
        "Unit": method_info["unit"],
        "Swiss score": category_scores["Swiss heated greenhouse"],
        "Spanish score": category_scores["Spanish greenhouse"],
        "Larger result": larger_system,
        "Larger / smaller": ratio,
    })

score_comparison = pd.DataFrame(comparison_rows)
display(score_comparison.style.format({
    "Swiss score": "{:.3g}",
    "Spanish score": "{:.3g}",
    "Larger / smaller": "{:.1f}×",
}))

### Reading the comparison

The ratio answers a within-category question: **how many times larger is the larger tomato result?** It does not compare AWARE with IBIF.

Expected pattern:

- Spanish greenhouse tomatoes have a moderately higher AWARE score.
- Swiss heated-greenhouse tomatoes have a much higher IBIF land-use score.

The next sections investigate which processes and locations create these differences.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (impact_label, method_info) in zip(axes, methods.items()):
    category_scores = scores[scores["Impact category"] == impact_label]
    category_scores = category_scores.set_index("Tomato system").loc[list(tomatoes)]

    bars = ax.bar(
        ["Switzerland", "Spain"],
        category_scores["Score"],
        color=method_info["color"],
        width=0.65,
    )
    ax.bar_label(bars, labels=[f"{value:.3g}" for value in category_scores["Score"]], padding=3)
    ax.set_title(impact_label)
    ax.set_ylabel(f"{method_info['unit']} per kg tomatoes")
    ax.set_ylim(0, category_scores["Score"].max() * 1.18)

fig.suptitle("Regionalised impacts of the two tomato systems", y=1.03)
plt.tight_layout()
plt.show()

## 4) Identify the processes that drive each score

A total score tells us **how much**, but not **why**. We therefore group characterized contributions by consumer process and location, then rank them by absolute contribution.

For AWARE, a contribution can be positive or negative because water consumption and releases can offset each other. Orange bars below indicate negative contributions.

In [ ]:
from textwrap import shorten


def impact_column(table):
    return "impact (mean)" if "impact (mean)" in table.columns else "impact"


def top_process_contributions(table, top_n=5):
    column = impact_column(table)
    grouped = (
        table.groupby(["consumer name", "consumer location"], dropna=False)[column]
        .sum()
        .reset_index(name="Contribution")
    )
    grouped["Absolute contribution"] = grouped["Contribution"].abs()
    total_absolute = grouped["Absolute contribution"].sum()
    grouped["Share of absolute total (%)"] = 100 * grouped["Absolute contribution"] / total_absolute
    grouped["Process"] = grouped["consumer name"].fillna("Unknown")
    grouped["Location"] = grouped["consumer location"].fillna("n/a")
    grouped["Label"] = [
        f"{shorten(process, width=45, placeholder='…')} ({location})"
        for process, location in zip(grouped["Process"], grouped["Location"])
    ]
    return grouped.nlargest(top_n, "Absolute contribution")


top_contributions = {
    key: top_process_contributions(table)
    for key, table in cf_tables.items()
}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for row, (impact_label, method_info) in enumerate(methods.items()):
    for column, tomato_label in enumerate(tomatoes):
        ax = axes[row, column]
        plot_data = top_contributions[(tomato_label, impact_label)].sort_values("Absolute contribution")
        colors = [method_info["color"] if value >= 0 else "#d95f02" for value in plot_data["Contribution"]]

        ax.barh(plot_data["Label"], plot_data["Contribution"], color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{impact_label}\n{tomato_label}")
        ax.set_xlabel(method_info["unit"])
        ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()

## 5) Put the location pattern on a map

The ranked country plots show which locations matter most, while the maps add geographic context.

- Subnational codes such as `US-CA` are aggregated to their parent country.
- AWARE uses a **diverging scale** centred on zero, so positive and negative contributions remain visible.
- Both tomato systems use the **same scale within each impact category**, allowing a visual comparison.

In [ ]:
import country_converter as coco
import plotly.express as px


def parent_country(location):
    code = str(location).split("-")[0]
    return code if len(code) == 2 and code.isalpha() else None


def group_location_contributions(table):
    column = impact_column(table)
    working = table[["consumer location", column]].copy()
    working["Country"] = working["consumer location"].map(parent_country)

    total_absolute = working[column].abs().sum()
    mapped_absolute = working.loc[working["Country"].notna(), column].abs().sum()
    coverage = 100 * mapped_absolute / total_absolute if total_absolute else 0.0

    grouped = working.dropna(subset=["Country"]).groupby("Country")[column].sum().reset_index()
    grouped = grouped.rename(columns={column: "Contribution"})
    grouped["Absolute contribution"] = grouped["Contribution"].abs()
    grouped_absolute = grouped["Absolute contribution"].sum()
    grouped["Share of absolute mapped total (%)"] = 100 * grouped["Absolute contribution"] / grouped_absolute
    grouped["ISO3"] = coco.convert(names=grouped["Country"].tolist(), to="ISO3", not_found=None)
    grouped = grouped[grouped["ISO3"].astype(str).str.len() == 3]
    return grouped, coverage


location_frames = []
coverage_rows = []
for (tomato_label, impact_label), table in cf_tables.items():
    location_frame, coverage = group_location_contributions(table)
    location_frame.insert(0, "Impact category", impact_label)
    location_frame.insert(0, "Tomato system", tomato_label)
    location_frames.append(location_frame)
    coverage_rows.append({"Tomato system": tomato_label, "Impact category": impact_label, "Mapped coverage (%)": coverage})

location_contributions = pd.concat(location_frames, ignore_index=True)
location_coverage = pd.DataFrame(coverage_rows)

In [ ]:
top_location_frames = []
for impact_label in methods:
    for tomato_label in tomatoes:
        subset = location_contributions[
            (location_contributions["Impact category"] == impact_label)
            & (location_contributions["Tomato system"] == tomato_label)
        ]
        top_location_frames.append(subset.nlargest(5, "Absolute contribution"))

top_locations = pd.concat(top_location_frames, ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for row, (impact_label, method_info) in enumerate(methods.items()):
    for column, tomato_label in enumerate(tomatoes):
        ax = axes[row, column]
        plot_data = top_locations[
            (top_locations["Impact category"] == impact_label)
            & (top_locations["Tomato system"] == tomato_label)
        ].sort_values("Absolute contribution")
        colors = [method_info["color"] if value >= 0 else "#d95f02" for value in plot_data["Contribution"]]

        ax.barh(plot_data["Country"], plot_data["Contribution"], color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{impact_label}\n{tomato_label}")
        ax.set_xlabel(method_info["unit"])

plt.tight_layout()
plt.show()

minimum_coverage = location_coverage["Mapped coverage (%)"].min()
print(f"Country mapping coverage is at least {minimum_coverage:.1f}% for every case.")

In [ ]:
for impact_label, method_info in methods.items():
    map_data = location_contributions[
        location_contributions["Impact category"] == impact_label
    ].copy()

    if impact_label == "AWARE water scarcity":
        limit = map_data["Contribution"].abs().max()
        color_scale = "RdBu_r"
        color_range = (-limit, limit)
        midpoint = 0
    else:
        color_scale = "Greens"
        color_range = (0, map_data["Contribution"].max())
        midpoint = None

    fig = px.choropleth(
        map_data,
        locations="ISO3",
        color="Contribution",
        facet_col="Tomato system",
        hover_name="Country",
        hover_data={"Contribution": ":.3g", "ISO3": False},
        projection="natural earth",
        color_continuous_scale=color_scale,
        range_color=color_range,
        color_continuous_midpoint=midpoint,
        title=f"{impact_label}: contribution by country",
    )
    fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
    fig.update_geos(showcoastlines=True, showcountries=True)
    fig.update_layout(
        width=1050,
        height=440,
        margin=dict(l=10, r=10, t=70, b=10),
        coloraxis_colorbar=dict(title=method_info["unit"]),
    )
    fig.show()

### Reading the maps

- In the **AWARE** map, red countries add to the score and blue countries subtract from it. The common scale makes the stronger Spanish contribution immediately visible.
- In the **IBIF land-use** map, much of the burden occurs upstream. Sweden, Germany, and Switzerland are important for the Swiss greenhouse system, even though the functional unit is tomatoes.
- The coverage table reports how much of the absolute contribution could be assigned to a country. This is more informative than printing a long list of unsupported regional codes.

A map shows *where* impacts occur; the ranked contribution table remains the better tool for reading exact values.

## 6) Decompose IBIF vertebrate impacts by pathway

The earlier IBIF method represents **overall biodiversity** and land use only. We now switch to the **vertebrate** species group and compare three available pressure pathways:

- climate change (`CO2`)
- land use (`LU`)
- roads

We also calculate the combined `all pressures` method and verify that it equals the sum of the available component methods for these datasets. Because the species group changes, these scores should not be compared directly with the earlier overall-biodiversity score.

In [ ]:
component_order = ["Climate change (CO2)", "Land use", "Roads"]
vertebrate_methods = {
    "Climate change (CO2)": ("IBIF", "biodiversity", "CO2", "vertebrates"),
    "Land use": ("IBIF", "biodiversity", "LU", "vertebrates"),
    "Roads": ("IBIF", "biodiversity", "roads", "vertebrates"),
    "All pressures": ("IBIF", "biodiversity", "all pressures", "vertebrates"),
}

missing = [method for method in vertebrate_methods.values() if method not in available_methods]
if missing:
    raise ValueError(f"Missing vertebrate IBIF methods: {missing}")

pathway_analyses = {}
pathway_rows = []
for pathway_label, method in vertebrate_methods.items():
    for tomato_label, activity in tomatoes.items():
        print(f"Calculating {pathway_label} for {tomato_label} ...")
        analysis = run_edges(activity, method)
        pathway_analyses[(tomato_label, pathway_label)] = analysis
        pathway_rows.append({
            "Tomato system": tomato_label,
            "Pathway": pathway_label,
            "Score": float(analysis.score),
        })

pathway_scores = pd.DataFrame(pathway_rows)

In [ ]:
component_scores = pathway_scores[pathway_scores["Pathway"].isin(component_order)].copy()
component_scores["Share of component sum (%)"] = (
    100 * component_scores["Score"]
    / component_scores.groupby("Tomato system")["Score"].transform("sum")
)

component_totals = (
    component_scores.groupby("Tomato system", as_index=False)["Score"]
    .sum()
    .rename(columns={"Score": "Sum of components"})
)
combined_scores = (
    pathway_scores[pathway_scores["Pathway"] == "All pressures"]
    [["Tomato system", "Score"]]
    .rename(columns={"Score": "All pressures"})
)
pathway_check = component_totals.merge(combined_scores, on="Tomato system")
pathway_check["Difference"] = pathway_check["Sum of components"] - pathway_check["All pressures"]
pathway_check["Matches?"] = np.isclose(
    pathway_check["Sum of components"], pathway_check["All pressures"]
)

display(component_scores.style.format({"Score": "{:.3g}", "Share of component sum (%)": "{:.1f}"}))
display(pathway_check.style.format({
    "Sum of components": "{:.3g}", "All pressures": "{:.3g}", "Difference": "{:.2g}",
}))

In [ ]:
pathway_colors = {
    "Climate change (CO2)": "#9467bd",
    "Land use": "#2e8b57",
    "Roads": "#8c564b",
}

composition = component_scores.pivot(index="Tomato system", columns="Pathway", values="Score")
composition = composition.loc[list(tomatoes), component_order]
composition_pct = 100 * composition.div(composition.sum(axis=1), axis=0)
totals = pathway_check.set_index("Tomato system").loc[list(tomatoes), "All pressures"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
composition_pct.plot(
    kind="bar", stacked=True, ax=axes[0], width=0.65,
    color=[pathway_colors[pathway] for pathway in component_order],
)
axes[0].set_title("Composition of vertebrate impacts")
axes[0].set_ylabel("Share of component sum (%)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Pathway", bbox_to_anchor=(1.02, 1), loc="upper left")

bars = axes[1].bar(["Switzerland", "Spain"], totals, color="#4c78a8", width=0.65)
axes[1].bar_label(bars, labels=[f"{value:.3g}" for value in totals], padding=3)
axes[1].set_title("Total vertebrate impact")
axes[1].set_ylabel("MSA-loss·km²·yr per kg tomatoes")
axes[1].set_ylim(0, totals.max() * 1.18)

plt.tight_layout()
plt.show()

print(f"Swiss / Spanish total vertebrate impact: {totals.iloc[0] / totals.iloc[1]:.1f}×")

### Which processes drive the pathways?

Instead of printing thousands of matched CF rows, we aggregate by process and location. Each subplot shows the three largest absolute contributions; the percentage at the end of each bar is its share of the pathway's absolute total.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 10))

for row, pathway_label in enumerate(component_order):
    for column, tomato_label in enumerate(tomatoes):
        analysis = pathway_analyses[(tomato_label, pathway_label)]
        table = analysis.generate_cf_table(split_aggregate_consumers=True)
        plot_data = top_process_contributions(table, top_n=3).sort_values("Absolute contribution")
        ax = axes[row, column]

        bar_positions = range(len(plot_data))
        bars = ax.barh(bar_positions, plot_data["Contribution"], color=pathway_colors[pathway_label])
        ax.set_yticks(bar_positions, labels=plot_data["Label"])
        labels = [f"{share:.0f}%" for share in plot_data["Share of absolute total (%)"]]
        ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.margins(x=0.22)
        ax.set_title(f"{pathway_label}\n{tomato_label}")
        ax.set_xlabel("MSA-loss·km²·yr")
        ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()

## 7) Generate a Sankey diagram

`SupplyChain` gives a complementary view of the technosphere pathways behind a score. Below we build a Sankey for the **Swiss heated-greenhouse tomato dataset** under **IBIF land use**, using a **1% cutoff** of the total score.

In [ ]:
from edges.supply_chain import SupplyChain

sankey_dataset_label = "Swiss heated greenhouse"
sankey_activity = tomatoes[sankey_dataset_label]

ibif_supply_chain = SupplyChain(
    activity=sankey_activity,
    method=ibif_landuse_method,
    amount=1,
    level=5,
    cutoff=0.01,
    cutoff_basis="total",
)

sankey_total_score = ibif_supply_chain.bootstrap()
sankey_df, sankey_total_score, sankey_reference_amount = ibif_supply_chain.calculate()

display(sankey_df[["level", "name", "location", "amount", "score", "share_of_total"]].head(12))
print("Sankey root dataset:", sankey_dataset_label)
print("Reference amount:", sankey_reference_amount)
print("Total IBIF score:", sankey_total_score, methods["IBIF land use"]["unit"])

In [ ]:
sankey_fig = ibif_supply_chain.plot_sankey(
    sankey_df,
    width_max=1600,
    height_max=700,
    auto_width=True,
)
sankey_fig

## 8) Reuse one `EdgeLCIA` object with `redo_lcia()`

Here we keep one method fixed, pick five random BAFU datasets, and calculate the same scores two ways:

1. create a fresh `EdgeLCIA` object for each dataset;
2. create one `EdgeLCIA` object, then switch demands with `redo_lcia()`.

The scores should be the same; only the runtime should change.

In [ ]:
import time

redo_method = aware_method
redo_sample_size = 5
rng = np.random.default_rng(42)

activity_pool = sorted(
    list(db),
    key=lambda activity: (
        activity["name"],
        activity.get("location") or "",
        activity.get("reference product") or "",
    ),
)
redo_activities = [
    activity_pool[index]
    for index in rng.choice(len(activity_pool), size=redo_sample_size, replace=False)
]

fresh_scores = []
fresh_seconds = []
for activity in redo_activities:
    start = time.perf_counter()
    fresh_scores.append(float(run_edges(activity, redo_method).score))
    fresh_seconds.append(time.perf_counter() - start)

reuse_scores = []
reuse_seconds = []
reuse_lca = None
for activity in redo_activities:
    start = time.perf_counter()
    if reuse_lca is None:
        reuse_lca = run_edges(activity, redo_method)
    else:
        reuse_lca.redo_lcia({activity: 1}, recompute_score=True)
    reuse_scores.append(float(reuse_lca.score))
    reuse_seconds.append(time.perf_counter() - start)

np.testing.assert_allclose(fresh_scores, reuse_scores, rtol=1e-10, atol=1e-12)

redo_table = pd.DataFrame({
    "Dataset": [activity["name"] for activity in redo_activities],
    "Location": [activity.get("location") for activity in redo_activities],
    "Fresh score": fresh_scores,
    "Reused score": reuse_scores,
    "Fresh seconds": fresh_seconds,
    "Reused seconds": reuse_seconds,
})
redo_table["Speed-up"] = redo_table["Fresh seconds"] / redo_table["Reused seconds"]

display(redo_table.style.format({
    "Fresh score": "{:.3g}", "Reused score": "{:.3g}",
    "Fresh seconds": "{:.2f}", "Reused seconds": "{:.2f}", "Speed-up": "{:.1f}×",
}))
print(f"Overall speed-up: {sum(fresh_seconds) / sum(reuse_seconds):.1f}×")

## Optional appendix: split aggregate locations for reporting

This advanced diagnostic is not needed to understand the main results. It shows how `edges` can replace an aggregate consumer location such as `RER` with country-level rows while preserving the total score.

We keep only the largest locations and a short preview of the `RER` shares.

In [ ]:
demo_key = ("Swiss heated greenhouse", "AWARE water scarcity")
demo_analysis = analyses[demo_key]
unsplit_table = demo_analysis.generate_cf_table()
split_table = cf_tables[demo_key]


def impact_by_location(table):
    column = impact_column(table)
    return table.groupby("consumer location")[column].sum()


split_comparison = pd.concat(
    {
        "Before split": impact_by_location(unsplit_table),
        "After split": impact_by_location(split_table),
    },
    axis=1,
).fillna(0)
split_comparison["Largest absolute value"] = split_comparison.abs().max(axis=1)
split_comparison = split_comparison.nlargest(8, "Largest absolute value").drop(columns="Largest absolute value")

print("Total preserved:", np.isclose(
    impact_by_location(unsplit_table).sum(),
    impact_by_location(split_table).sum(),
))
display(split_comparison.style.format("{:.3g}"))

ax = split_comparison.plot(kind="bar", figsize=(9, 4), color=["#9ecae1", "#2171b5"])
ax.set_ylabel(methods["AWARE water scarcity"]["unit"])
ax.set_xlabel("Consumer location")
ax.set_title("Aggregate location before and after country-level reporting split")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

rer_shares = []
for scenario_cf in demo_analysis.scenario_cfs:
    if scenario_cf.get("consumer", {}).get("location") != "RER":
        continue
    for component in scenario_cf.get("reporting_split", []):
        rer_shares.append({
            "Country": component["consumer_location"],
            "Share": component["share"],
        })

rer_share_table = pd.DataFrame(rer_shares).drop_duplicates().nlargest(8, "Share")
display(rer_share_table.style.format({"Share": "{:.1%}"}))

## Checkpoint

- Which tomato system has the higher AWARE score, and by how much?
- Which system has the higher IBIF land-use score?
- Why can an AWARE country contribution be negative?
- Which upstream countries drive the Swiss IBIF land-use result?
- Are the vertebrate pathway shares similar even though the total scores differ strongly?
- What does the Sankey add to the ranked contribution table?

## Recap

After this notebook, you should now be able to:

- compare two products within one regionalised impact category using the correct unit
- distinguish a total score from contribution-based explanations
- read ranked process and country contributions
- interpret shared-scale maps, including signed AWARE contributions
- compare pathway composition separately from total magnitude
- verify a combined method against the sum of its component methods
- use a Sankey for a complementary supply-chain view
- use `redo_lcia()` when the method stays fixed and only the demand changes